In [1]:
%cd ../src
%load_ext autoreload
%autoreload 2

/Users/martinbierey/git/dax-hoc/src


### 1) Preparation 

In [1]:
import logging
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path
from sqlalchemy import create_engine, select
from db import bootstrap, get_engine, get_session, read_all 
from db.models import Company, EarningsEvent, NewsItem
from llm.registry import call_llm
import json
from ensemble.interpreter import _PROMPT

from db.models import Company, EarningsEvent, ExpectationsSnapshot
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s — %(message)s")
logger = logging.getLogger(__name__)

NEWS_ITEM_ID = "e1ec472f-4aed-4662-a5d3-c9cf3f9d5c67_en"

/Users/martinbierey/git/dax-hoc/.venv/lib/python3.14/site-packages/anthropic/_compat.py:48: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.typing import (


In [2]:
# ── load NewsItem from real DB ────────────────────────────────────────────────
real_engine = get_engine()
session = get_session(real_engine)
try:
    news_item = session.scalars(select(NewsItem).where(NewsItem.id == NEWS_ITEM_ID)).first()
    if news_item is None:
        logger.error("NewsItem not found. Run scripts/fetch_auto1_announcement.py first.")
        sys.exit(1)
    # Detach from session so we can use it freely
    session.expunge(news_item)
finally:
    session.close()

logger.info("Loaded NewsItem: %s", news_item.headline)

INFO __main__ — Loaded NewsItem: AUTO1 Group achieves record-breaking 2025 results, delivering all-time highs in units sold, gross profit and adjusted EBITDA


In [25]:
company = Company(
    isin="DE000A2LQ884",
    name="AUTO1 Group SE",
    index_name="MDAX",
    industry="Automotive E-Commerce",
    description="Europe's leading digital automotive marketplace for used cars.",
    synced_at=datetime.now(timezone.utc),
)

event = EarningsEvent(
    id=str(uuid.uuid4()),
    isin="DE000A2LQ884",
    fiscal_period="2026-Q1",
    event_type="quarterly",
    actual_release_at=datetime(2026, 2, 25, 7, 30, tzinfo=timezone.utc),
    status="released",
)